# Day 1: Project Structure & Experiment Tracking

**MLOps in Practice Workshop**

---

## Learning Objectives

By the end of this session, you will be able to:

1. Explain why notebooks break in production and how MLOps addresses this
2. Structure ML projects using industry-standard templates
3. Track experiments with MLflow (parameters, metrics, artifacts)
4. Version data with DVC
5. Manage configurations with Hydra/OmegaConf

---

## 1. The MLOps Problem: Why Notebooks Break in Production

Jupyter notebooks are excellent for exploration and prototyping, but they introduce serious problems when used as production artifacts:

### Common Failure Modes

| Problem | Impact |
|---------|--------|
| **Hidden state** | Cells executed out of order produce different results |
| **No modularity** | Code cannot be imported, tested, or reused |
| **No versioning** | JSON diffs are unreadable; merge conflicts are painful |
| **Hardcoded paths** | `pd.read_csv('/home/alice/data.csv')` fails everywhere else |
| **No dependency management** | "It works on my machine" |
| **Manual execution** | Someone has to click "Run All" every morning |
| **No experiment tracking** | Which hyperparameters produced the best model? |

### The MLOps Solution

MLOps applies DevOps principles to machine learning:

- **Reproducibility** through version control, containers, and pipelines
- **Automation** through CI/CD and orchestration
- **Monitoring** through drift detection and alerting
- **Collaboration** through standardized project structure and experiment tracking

This workshop walks through these pillars over four days.

---

## 2. Project Structure Best Practices

### The Cookiecutter Data Science Template

The most widely adopted ML project template:

```
my_ml_project/
|
+-- data/
|   +-- raw/              # Immutable original data
|   +-- processed/        # Cleaned, transformed data
|   +-- external/         # Third-party data sources
|
+-- models/               # Trained and serialized models
|
+-- notebooks/            # Exploration only (numbered: 01_, 02_)
|
+-- src/
|   +-- __init__.py
|   +-- data/             # Data loading and processing
|   |   +-- make_dataset.py
|   +-- features/         # Feature engineering
|   |   +-- build_features.py
|   +-- models/           # Training and prediction
|   |   +-- train.py
|   |   +-- predict.py
|   +-- evaluation/       # Metrics and evaluation
|       +-- evaluate.py
|
+-- tests/                # Unit and integration tests
|   +-- test_data.py
|   +-- test_model.py
|
+-- configs/              # Hydra configuration files
|   +-- config.yaml
|   +-- model/
|   +-- data/
|
+-- Dockerfile
+-- docker-compose.yml
+-- requirements.txt
+-- setup.py
+-- dvc.yaml              # DVC pipeline definition
+-- params.yaml           # DVC parameters
+-- README.md
```

### Key Principles

1. **Data is immutable** -- never modify raw data in place
2. **Notebooks are for exploration** -- production code lives in `src/`
3. **Configuration is separate from code** -- use YAML files, not hardcoded values
4. **Everything is version-controlled** -- code in Git, data in DVC

---

## 3. MLflow Experiment Tracking

MLflow is an open-source platform for managing the ML lifecycle. We focus on the **Tracking** component: logging parameters, metrics, and artifacts for every experiment run.

In [ ]:
# Install dependencies (run once)
# !pip install mlflow scikit-learn pandas numpy

In [ ]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings("ignore")

print(f"MLflow version: {mlflow.__version__}")

In [ ]:
# Load and prepare data
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Features: {X.shape[1]}")
print(f"Classes: {np.unique(y)}")

### 3.1 Logging Parameters, Metrics, and Artifacts

In [ ]:
# Set up MLflow experiment
mlflow.set_experiment("wine-classification")

def train_and_log(n_estimators, max_depth, min_samples_split):
    """Train a model and log everything to MLflow."""
    with mlflow.start_run():
        # Log parameters
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("min_samples_split", min_samples_split)
        mlflow.log_param("model_type", "RandomForest")

        # Train model
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            random_state=42,
        )
        model.fit(X_train, y_train)

        # Predict and evaluate
        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average="weighted")

        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("f1_weighted", f1)

        # Log the model as an artifact
        mlflow.sklearn.log_model(model, "model")

        # Log feature importances as a CSV artifact
        importance_df = pd.DataFrame({
            "feature": wine.feature_names,
            "importance": model.feature_importances_,
        }).sort_values("importance", ascending=False)

        importance_path = "feature_importances.csv"
        importance_df.to_csv(importance_path, index=False)
        mlflow.log_artifact(importance_path)

        print(
            f"Run logged | n_estimators={n_estimators}, "
            f"max_depth={max_depth}, "
            f"accuracy={accuracy:.4f}, f1={f1:.4f}"
        )

        return accuracy, f1

### 3.2 Comparing Runs -- Hyperparameter Sweep

In [ ]:
# Run multiple experiments with different hyperparameters
configs = [
    {"n_estimators": 50,  "max_depth": 3,    "min_samples_split": 2},
    {"n_estimators": 100, "max_depth": 5,    "min_samples_split": 2},
    {"n_estimators": 200, "max_depth": 10,   "min_samples_split": 5},
    {"n_estimators": 200, "max_depth": None,  "min_samples_split": 2},
    {"n_estimators": 500, "max_depth": None,  "min_samples_split": 10},
]

results = []
for cfg in configs:
    acc, f1 = train_and_log(**cfg)
    results.append({**cfg, "accuracy": acc, "f1_weighted": f1})

results_df = pd.DataFrame(results)
print("\n=== All Runs ===")
print(results_df.to_string(index=False))

In [ ]:
# Query runs programmatically from MLflow
experiment = mlflow.get_experiment_by_name("wine-classification")
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.accuracy DESC"],
)

print("Top 3 runs by accuracy:")
display_cols = [
    "run_id",
    "params.n_estimators",
    "params.max_depth",
    "metrics.accuracy",
    "metrics.f1_weighted",
]
print(runs[display_cols].head(3).to_string(index=False))

### 3.3 Loading a Logged Model

MLflow lets you reload any logged model by its run ID.

In [ ]:
# Load the best model from MLflow
best_run_id = runs.iloc[0]["run_id"]
loaded_model = mlflow.sklearn.load_model(f"runs:/{best_run_id}/model")

# Verify predictions match
y_pred_loaded = loaded_model.predict(X_test)
print(f"Best run ID: {best_run_id}")
print(f"Accuracy from loaded model: {accuracy_score(y_test, y_pred_loaded):.4f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred_loaded, target_names=wine.target_names)}")

---

## 4. DVC Basics: Data Version Control

DVC tracks large files and datasets alongside your Git repository. It replaces large files with small `.dvc` pointer files in Git, while the actual data is stored in a remote (S3, GCS, local, etc.).

### 4.1 Core DVC Commands

```bash
# Initialize DVC in an existing Git repo
dvc init

# Track a data file
dvc add data/raw/dataset.csv
# This creates data/raw/dataset.csv.dvc (pointer file)
# and adds data/raw/dataset.csv to .gitignore

# Commit the pointer file to Git
git add data/raw/dataset.csv.dvc data/raw/.gitignore
git commit -m "Track raw dataset with DVC"

# Set up a remote storage
dvc remote add -d myremote s3://my-bucket/dvc-storage

# Push data to remote
dvc push

# Pull data on another machine
dvc pull
```

### 4.2 DVC Pipelines

DVC pipelines define reproducible workflows in a `dvc.yaml` file:

```yaml
# dvc.yaml
stages:
  prepare:
    cmd: python src/data/make_dataset.py
    deps:
      - src/data/make_dataset.py
      - data/raw/dataset.csv
    outs:
      - data/processed/train.csv
      - data/processed/test.csv

  train:
    cmd: python src/models/train.py
    deps:
      - src/models/train.py
      - data/processed/train.csv
    params:
      - model.n_estimators
      - model.max_depth
    outs:
      - models/model.pkl
    metrics:
      - reports/metrics.json:
          cache: false

  evaluate:
    cmd: python src/evaluation/evaluate.py
    deps:
      - src/evaluation/evaluate.py
      - models/model.pkl
      - data/processed/test.csv
    metrics:
      - reports/eval_metrics.json:
          cache: false
    plots:
      - reports/confusion_matrix.csv:
          cache: false
```

```bash
# Run the full pipeline
dvc repro

# Compare metrics across Git branches/tags
dvc metrics diff

# Visualize pipeline DAG
dvc dag
```

---

## 5. Configuration Management with Hydra / OmegaConf

Hydra lets you compose configurations from multiple YAML files and override them from the command line. OmegaConf provides the underlying config system.

In [ ]:
# Install: pip install hydra-core omegaconf
from omegaconf import OmegaConf

# Define configuration as a structured dictionary
config_dict = {
    "data": {
        "raw_path": "data/raw/wine.csv",
        "processed_path": "data/processed/",
        "test_size": 0.2,
        "random_state": 42,
    },
    "model": {
        "type": "RandomForest",
        "n_estimators": 200,
        "max_depth": 10,
        "min_samples_split": 5,
    },
    "training": {
        "experiment_name": "wine-classification",
        "mlflow_tracking_uri": "sqlite:///mlflow.db",
    },
}

cfg = OmegaConf.create(config_dict)
print(OmegaConf.to_yaml(cfg))

In [ ]:
# Access config values with dot notation
print(f"Model type: {cfg.model.type}")
print(f"Test size: {cfg.data.test_size}")
print(f"N estimators: {cfg.model.n_estimators}")

# Merge overrides (simulates command-line overrides)
overrides = OmegaConf.create({
    "model": {
        "n_estimators": 500,
        "max_depth": None,
    }
})
merged_cfg = OmegaConf.merge(cfg, overrides)

print(f"\nAfter override:")
print(f"  n_estimators: {merged_cfg.model.n_estimators}")
print(f"  max_depth: {merged_cfg.model.max_depth}")

In [ ]:
# Using config with MLflow
def train_with_config(cfg):
    """Train a model using a structured config object."""
    mlflow.set_experiment(cfg.training.experiment_name)

    with mlflow.start_run(run_name="config-driven-run"):
        # Log entire config as params
        mlflow.log_params({
            "model_type": cfg.model.type,
            "n_estimators": cfg.model.n_estimators,
            "max_depth": str(cfg.model.max_depth),
            "min_samples_split": cfg.model.min_samples_split,
            "test_size": cfg.data.test_size,
        })

        # Save config as artifact
        config_path = "run_config.yaml"
        OmegaConf.save(cfg, config_path)
        mlflow.log_artifact(config_path)

        # Train
        model = RandomForestClassifier(
            n_estimators=cfg.model.n_estimators,
            max_depth=cfg.model.max_depth,
            min_samples_split=cfg.model.min_samples_split,
            random_state=cfg.data.random_state,
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average="weighted")

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("f1_weighted", f1)
        mlflow.sklearn.log_model(model, "model")

        print(f"Config-driven run: accuracy={accuracy:.4f}, f1={f1:.4f}")

train_with_config(merged_cfg)

### Hydra in Production Scripts

In a real project, your training script would use the `@hydra.main` decorator:

```python
# src/models/train.py
import hydra
from omegaconf import DictConfig

@hydra.main(config_path="../../configs", config_name="config", version_base=None)
def train(cfg: DictConfig):
    print(f"Training {cfg.model.type} with n_estimators={cfg.model.n_estimators}")
    # ... training logic ...

if __name__ == "__main__":
    train()
```

Then run from the command line with overrides:

```bash
python src/models/train.py model.n_estimators=500 model.max_depth=20
```

---

## 6. Lab: Refactor a Messy Notebook into a Clean Project

Below is a typical "messy notebook" -- all logic crammed into a single script with hardcoded values, no logging, and no structure. Your task is to refactor it.

### The Messy Code

In [ ]:
# ============================================================
# THE MESSY NOTEBOOK (Do Not Use in Production!)
# ============================================================
# Everything is in one cell, hardcoded, no tracking, no config.

import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
import pickle

# Load data (hardcoded)
data = load_iris()
X = data.data
y = data.target

# Split (magic numbers)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train (hardcoded hyperparameters, no logging)
model = GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1)
model.fit(X_train, y_train)

# Evaluate (just a print)
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

# Save (hardcoded path, no versioning)
with open("my_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Done!")

### Your Task

Refactor the messy code above into a clean, production-ready structure:

1. **Create a config** using OmegaConf with all hyperparameters and paths
2. **Extract functions** for data loading, preprocessing, training, and evaluation
3. **Add MLflow tracking** for parameters, metrics, and the model artifact
4. **Remove hardcoded values** -- everything should come from the config

Use the cell below for your solution.

In [ ]:
# ============================================================
# LAB SOLUTION TEMPLATE
# ============================================================
# Fill in the functions below to refactor the messy code.

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from omegaconf import OmegaConf
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score


# Step 1: Define config
cfg = OmegaConf.create({
    "data": {
        "test_size": 0.3,
        "random_state": 42,
    },
    "model": {
        "n_estimators": 100,
        "max_depth": 3,
        "learning_rate": 0.1,
    },
    "experiment": {
        "name": "iris-refactored",
    },
})


# Step 2: Extract functions
def load_data():
    """Load and return the iris dataset."""
    # TODO: implement
    pass


def preprocess(X_train, X_test):
    """Scale features using StandardScaler."""
    # TODO: implement
    pass


def train_model(X_train, y_train, cfg):
    """Train a GradientBoostingClassifier with config params."""
    # TODO: implement
    pass


def evaluate_model(model, X_test, y_test):
    """Evaluate model and return metrics dict."""
    # TODO: implement
    pass


# Step 3: Main pipeline with MLflow tracking
def run_pipeline(cfg):
    """Execute the full training pipeline with MLflow logging."""
    mlflow.set_experiment(cfg.experiment.name)

    with mlflow.start_run():
        # TODO: load data, preprocess, train, evaluate
        # TODO: log params, metrics, and model to MLflow
        pass


# Run it
# run_pipeline(cfg)

---

## Summary

Today we covered the foundational pillars of MLOps:

| Topic | Key Takeaway |
|-------|-------------|
| **Notebooks in production** | They break due to hidden state, no modularity, and hardcoded values |
| **Project structure** | Use Cookiecutter templates; separate code, data, config, and notebooks |
| **MLflow** | Track every experiment: parameters, metrics, artifacts, and models |
| **DVC** | Version data and define reproducible pipelines with `dvc.yaml` |
| **Hydra/OmegaConf** | Externalize configuration; compose and override from the command line |

### Next Session

**Day 2: Docker & APIs** -- We will containerize ML models and serve them through FastAPI endpoints.